# Exploration et visualisation du Registre Parcellaire Graphique (RPG 2023)

**Projet Python pour la Data Science (ENSAE 2A)** — *Eau et agriculture : qualité des eaux et usage des pesticides*
**Source** : archive nationale du RPG 2023 publiée par l'IGN, réduite à une ligne par parcelle avec son département (`data/processed/parcelles_rpg_2023.parquet`, construite par `src/agriculture/construire_parcelles_rpg.py` et déposée dans la release).

---

## Objectif de ce notebook

Dans le cadre du projet croisant **qualité des eaux (Hub'Eau)**, **achats de produits phytosanitaires (BNV-D)** et **occupation agricole des sols (RPG)**, ce notebook explore le **RPG 2023**, l'année de référence retenue pour le projet, sur les 94 départements de métropole continentale (sans la Corse).

L'analyse porte sur quatre variables :
- **`surface_ha`** : surface de la parcelle (en hectares) ;
- **`libelle_categorie`** : grande catégorie d'occupation du sol (*Terres arables*, *Prairies permanentes*, *Cultures permanentes*, *Surfaces boisées*), tirée du code culture ;
- **`libelle_groupe`** : groupe de cultures du RPG (23 groupes) ;
- **`libelle_culture`** : culture principale de la parcelle (137 cultures, libellés de la nomenclature de l'IGN).

### Pourquoi cette exploration avant la fusion des trois bases ?
1. **Repérer l'exposition théorique aux pesticides par filière** : les traitements se concentrent sur les **terres arables** (céréales à paille, maïs, colza, betterave, pomme de terre) et les **cultures permanentes** (vignes, vergers), tandis que les **prairies permanentes** et les **estives** en reçoivent très peu.
2. **Caractériser la structure du parcellaire (`surface_ha`)** : l'opposition entre openfield céréalier (grandes parcelles) et zones d'élevage ou de viticulture (parcellaire morcelé) éclaire la géographie agricole française.

In [ ]:
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from IPython.display import display

# Positionnement automatique à la racine du dépôt (quel que soit le sous-dossier d'exécution, ex. notebooks/exploration/)
for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "src").is_dir():
        os.chdir(_parent)
        break

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
    "font.size": 10.5,
})

# Table de correspondance Département -> Région administrative (INSEE)
DEPARTEMENT_VERS_REGION = {
    # Auvergne-Rhône-Alpes
    "01": "Auvergne-Rhône-Alpes", "03": "Auvergne-Rhône-Alpes", "07": "Auvergne-Rhône-Alpes",
    "15": "Auvergne-Rhône-Alpes", "26": "Auvergne-Rhône-Alpes", "38": "Auvergne-Rhône-Alpes",
    "42": "Auvergne-Rhône-Alpes", "43": "Auvergne-Rhône-Alpes", "63": "Auvergne-Rhône-Alpes",
    "69": "Auvergne-Rhône-Alpes", "73": "Auvergne-Rhône-Alpes", "74": "Auvergne-Rhône-Alpes",
    # Bourgogne-Franche-Comté
    "21": "Bourgogne-Franche-Comté", "25": "Bourgogne-Franche-Comté", "39": "Bourgogne-Franche-Comté",
    "58": "Bourgogne-Franche-Comté", "70": "Bourgogne-Franche-Comté", "71": "Bourgogne-Franche-Comté",
    "89": "Bourgogne-Franche-Comté", "90": "Bourgogne-Franche-Comté",
    # Bretagne
    "22": "Bretagne", "29": "Bretagne", "35": "Bretagne", "56": "Bretagne",
    # Centre-Val de Loire
    "18": "Centre-Val de Loire", "28": "Centre-Val de Loire", "36": "Centre-Val de Loire",
    "37": "Centre-Val de Loire", "41": "Centre-Val de Loire", "45": "Centre-Val de Loire",
    # Grand Est
    "08": "Grand Est", "10": "Grand Est", "51": "Grand Est", "52": "Grand Est",
    "54": "Grand Est", "55": "Grand Est", "57": "Grand Est", "67": "Grand Est",
    "68": "Grand Est", "88": "Grand Est",
    # Hauts-de-France
    "02": "Hauts-de-France", "59": "Hauts-de-France", "60": "Hauts-de-France",
    "62": "Hauts-de-France", "80": "Hauts-de-France",
    # Île-de-France
    "75": "Île-de-France", "77": "Île-de-France", "78": "Île-de-France", "91": "Île-de-France",
    "92": "Île-de-France", "93": "Île-de-France", "94": "Île-de-France", "95": "Île-de-France",
    # Normandie
    "14": "Normandie", "27": "Normandie", "50": "Normandie", "61": "Normandie", "76": "Normandie",
    # Nouvelle-Aquitaine
    "16": "Nouvelle-Aquitaine", "17": "Nouvelle-Aquitaine", "19": "Nouvelle-Aquitaine",
    "23": "Nouvelle-Aquitaine", "24": "Nouvelle-Aquitaine", "33": "Nouvelle-Aquitaine",
    "40": "Nouvelle-Aquitaine", "47": "Nouvelle-Aquitaine", "64": "Nouvelle-Aquitaine",
    "79": "Nouvelle-Aquitaine", "86": "Nouvelle-Aquitaine", "87": "Nouvelle-Aquitaine",
    # Occitanie
    "09": "Occitanie", "11": "Occitanie", "12": "Occitanie", "30": "Occitanie",
    "31": "Occitanie", "32": "Occitanie", "34": "Occitanie", "46": "Occitanie",
    "48": "Occitanie", "65": "Occitanie", "66": "Occitanie", "81": "Occitanie", "82": "Occitanie",
    # Pays de la Loire
    "44": "Pays de la Loire", "49": "Pays de la Loire", "53": "Pays de la Loire",
    "72": "Pays de la Loire", "85": "Pays de la Loire",
    # Provence-Alpes-Côte d'Azur
    "04": "Provence-Alpes-Côte d'Azur", "05": "Provence-Alpes-Côte d'Azur",
    "06": "Provence-Alpes-Côte d'Azur", "13": "Provence-Alpes-Côte d'Azur",
    "83": "Provence-Alpes-Côte d'Azur", "84": "Provence-Alpes-Côte d'Azur",
}

from src.agriculture.constantes_rpg import CATEGORIES_CULTURE, LIBELLES_GROUPES_RPG, URL_NOMENCLATURE_CULTURES_2023
from src.recuperer_release import recuperer_release

# Parcelles du RPG 2023 tirées de l'archive nationale de l'IGN (une ligne par parcelle, avec son département) :
# lues dans la release (le calcul à partir de l'archive de 2,75 Go prend plus de 5 minutes)
t0 = time.time()
df = pd.read_parquet(recuperer_release("parcelles_rpg_2023.parquet"))

# Libellés : catégories et groupes (constantes du projet), cultures (nomenclature officielle de l'IGN)
nomenclature = pd.read_csv(URL_NOMENCLATURE_CULTURES_2023, sep=";", encoding="cp1252", dtype=str)
df["libelle_categorie"] = df["categorie_culture"].map(CATEGORIES_CULTURE)
df["libelle_groupe"] = df["code_groupe"].map(LIBELLES_GROUPES_RPG)
df["libelle_culture"] = df["code_culture"].map(dict(zip(nomenclature["CODE_CULTURE"], nomenclature["LIBELLE_CULTURE"])))

colonnes_utiles = [
    "code_dep", "nom_dep", "surface_ha",
    "categorie_culture", "libelle_categorie",
    "code_groupe", "libelle_groupe",
    "code_culture", "libelle_culture",
]

# Bilan qualité avant création des colonnes dérivées
bilan_colonnes = pd.DataFrame({
    "Type": df[colonnes_utiles].dtypes.astype(str),
    "Nb valeurs manquantes": df[colonnes_utiles].isna().sum(),
    "% manquantes": (100 * df[colonnes_utiles].isna().mean()).round(2),
    "Nb modalités uniques": [df[c].nunique(dropna=True) for c in colonnes_utiles],
})

# Les parcelles sans catégorie (surfaces non agricoles, bordures, bandes tampons) sont gardées dans une modalité explicite
df["libelle_categorie_propre"] = df["libelle_categorie"].fillna("Non attribué / Hors production (IAE, SNE)")
df["region"] = df["code_dep"].map(DEPARTEMENT_VERS_REGION).fillna("Autre")

duree = time.time() - t0
print(f"Chargement terminé en {duree:.2f} s : {len(df):,} parcelles réparties sur {df['code_dep'].nunique()} départements.")
print(f"Surface totale déclarée (SAU) : {df['surface_ha'].sum():,.1f} ha ({df['surface_ha'].sum() / 1e6:.2f} millions d'ha)")

display(bilan_colonnes)

### Analyse de la structure et de la qualité des données

1. **Couverture et volume** :
   - La table compte **9 742 907 parcelles** pour une surface déclarée de **27 457 975 ha** (~**27,46 millions d'hectares**) sur **94 départements**, ce qui correspond à la surface agricole déclarée à la PAC en métropole continentale. Les doublons et les parcelles de surface nulle (environ 8 500 miettes de moins de 0,005 ha) ont été retirés par `nettoyer_parcelles_rpg`.

2. **Complétude des variables** :
   - **`surface_ha`**, **`libelle_groupe`** (23 modalités) et **`libelle_culture`** (137 modalités) n'ont **aucune valeur manquante**.
   - Seule **`libelle_categorie`** a **965 178 valeurs manquantes (9,85 % des parcelles)** : ce sont surtout des surfaces non agricoles (`SNE`, 445 906 parcelles), des bandes tampons (`BTA`, 326 522) et des bordures de champs (`BOR`, 154 642), en moyenne de **0,25 ha**.
   - Nous ne supprimons pas ces lignes : elles forment une modalité explicite **`"Non attribué / Hors production (IAE, SNE)"`**, qui ne pèse que **0,88 % de la surface** (`243 794 ha`).

---

## 1. Grandes catégories d'occupation du sol (`libelle_categorie`) et taille des parcelles (`surface_ha`)

Comment l'espace agricole se répartit-il entre les grandes catégories, et comment la **taille des parcelles** varie-t-elle selon la catégorie ?

In [ ]:
# Statistiques descriptives de surface_ha par grande catégorie de culture
stats_cat = (
    df.groupby("libelle_categorie_propre", as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_totale_ha=("surface_ha", "sum"),
        moyenne_ha=("surface_ha", "mean"),
        q25_ha=("surface_ha", lambda s: s.quantile(0.25)),
        mediane_ha=("surface_ha", "median"),
        q75_ha=("surface_ha", lambda s: s.quantile(0.75)),
        q95_ha=("surface_ha", lambda s: s.quantile(0.95)),
        max_ha=("surface_ha", "max"),
    )
    .sort_values("surface_totale_ha", ascending=False)
)
stats_cat["part_surface_%"] = (100 * stats_cat["surface_totale_ha"] / df["surface_ha"].sum()).round(2)
stats_cat["part_parcelles_%"] = (100 * stats_cat["nb_parcelles"] / len(df)).round(2)

display(
    stats_cat[[
        "libelle_categorie_propre", "nb_parcelles", "part_parcelles_%",
        "surface_totale_ha", "part_surface_%",
        "moyenne_ha", "q25_ha", "mediane_ha", "q75_ha", "q95_ha", "max_ha"
    ]].round(2)
)

# Visualisation 1 : Poids des grandes catégories (gauche) & distribution de surface_ha (droite)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16.5, 6.4))

# Sous-graphique 1 : Part des surfaces vs Part du nombre de parcelles
cats_plot = stats_cat[stats_cat["libelle_categorie_propre"] != "Surfaces boisées"].copy()
y_pos = np.arange(len(cats_plot))
h_bar = 0.36

b1 = ax1.barh(
    y_pos - h_bar / 2,
    cats_plot["part_surface_%"],
    height=h_bar,
    color="#2b8cbe",
    label="Part de la surface totale (% SAU)",
)
b2 = ax1.barh(
    y_pos + h_bar / 2,
    cats_plot["part_parcelles_%"],
    height=h_bar,
    color="#fd8d3c",
    label="Part du nombre de parcelles (%)",
)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(cats_plot["libelle_categorie_propre"], fontweight="bold")
ax1.invert_yaxis()
ax1.set_xlabel("Pourcentage du total national (%)")
ax1.set_xlim(0, 72)
ax1.set_title("Grandes catégories : Surface (% SAU)\nvs Nombre de parcelles (%)", fontweight="bold", pad=12)
ax1.legend(loc="lower right", frameon=True, facecolor="white")

for bar in b1:
    w = bar.get_width()
    ax1.text(w + 0.8, bar.get_y() + bar.get_height() / 2, f"{w:.1f} %", va="center", fontsize=9.5, color="#1b5e85", fontweight="bold")
for bar in b2:
    w = bar.get_width()
    ax1.text(w + 0.8, bar.get_y() + bar.get_height() / 2, f"{w:.1f} %", va="center", fontsize=9.5, color="#b85814")

# Sous-graphique 2 : Distribution de la taille des parcelles (échelle log10) par catégorie
categories_principales = ["Terres arables", "Prairies permanentes", "Cultures permanentes"]
couleurs_cat = {"Terres arables": "#e34a33", "Prairies permanentes": "#31a354", "Cultures permanentes": "#756bb1"}

donnees_box = [
    df.loc[(df["libelle_categorie_propre"] == cat) & (df["surface_ha"] > 0), "surface_ha"].to_numpy()
    for cat in categories_principales
]

bp = ax2.boxplot(
    donnees_box,
    orientation="horizontal",
    patch_artist=True,
    showfliers=False,
    whis=(10, 90),
    medianprops={"color": "#111111", "linewidth": 2.0},
)

for patch, cat in zip(bp["boxes"], categories_principales):
    patch.set_facecolor(couleurs_cat[cat])
    patch.set_alpha(0.7)

ax2.set_yticks(range(1, len(categories_principales) + 1))
ax2.set_yticklabels(categories_principales, fontsize=9.5)
ax2.invert_yaxis()
ax2.set_xscale("log")
ax2.set_xlabel("Surface de la parcelle en ha (échelle log)\nboîte : Q25–Q75, moustaches : D1–D9")
ax2.set_title("Morphologie parcellaire (`surface_ha`)\npar grande catégorie", fontweight="bold", pad=12)

plt.tight_layout()
plt.show()

### Interprétation : répartition par grande catégorie et taille des parcelles

1. **Prédominance des Terres arables et des Prairies permanentes** :
   - Les **Terres arables (`TA`)** couvrent **16,69 millions d'hectares**, soit **60,8 % de la surface** (pour **48,2 %** des parcelles). C'est sur ce compartiment, soumis aux rotations annuelles et aux traitements herbicides, fongicides et insecticides, que se concentrent les achats de substances actives de la BNV-D.
   - Les **Prairies permanentes (`PP`)** forment le deuxième bloc avec **9,68 millions d'hectares** (**35,3 %** de la surface, **34,4 %** des parcelles).
   - Les **Cultures permanentes (`CP`, vignes, vergers, oliviers, fruits à coque)** ne pèsent que **3,1 % de la surface (`870 013 ha`)**, mais **7,6 % des parcelles (`748 700`)** : leur parcellaire est très morcelé.
   - Les *Surfaces boisées* sont anecdotiques (136 parcelles, 113 ha).

2. **Forte asymétrie de `surface_ha`** :
   - La parcelle médiane fait **1,24 ha** (Q25 = **0,39 ha**, Q75 = **3,21 ha**) pour une moyenne de **2,83 ha**, plus du double de la médiane.
   - Les **Terres arables** ont les plus grandes parcelles (médiane **2,00 ha**, Q75 = **4,47 ha**, Q95 = **12,22 ha**, moyenne **3,55 ha**), devant les **Prairies permanentes** (médiane **1,21 ha**, moyenne **2,94 ha**, maximum de **6 545 ha** pour les grandes estives collectives de montagne).
   - Les **Cultures permanentes** ont une médiane de **0,63 ha** seulement (moyenne **1,16 ha**), typique du morcellement viticole et arboricole.

---

## 2. Paysage cultural : groupes de cultures (`libelle_groupe`) et cultures principales (`libelle_culture`)

Affinons l'analyse au niveau des **23 groupes de cultures** et du **Top 15 des 137 cultures** du RPG 2023.

In [ ]:
# Agrégation par groupe de cultures (libelle_groupe)
stats_groupe = (
    df.groupby("libelle_groupe", as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_ha=("surface_ha", "sum"),
        categorie_dominante=("libelle_categorie_propre", lambda s: s.mode().iloc[0] if not s.mode().empty else "Autre"),
    )
    .sort_values("surface_ha", ascending=False)
)
stats_groupe["surface_mha"] = stats_groupe["surface_ha"] / 1e6
stats_groupe["part_sau_%"] = 100 * stats_groupe["surface_ha"] / df["surface_ha"].sum()

# Agrégation pour le Top 15 des cultures détaillées (libelle_culture)
top15_cult = df.groupby("libelle_culture")["surface_ha"].sum().nlargest(15) / 1e6

# Raccourcissement lisible des libellés très longs pour l'axe Y
labels_courts = [
    lbl[:52] + "…" if len(lbl) > 53 else lbl
    for lbl in top15_cult.index
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 8.5))

# Sous-graphique 1 : Les 23 groupes de cultures colorés par grande catégorie dominante
palette_cat = {
    "Terres arables": "#2b8cbe",
    "Prairies permanentes": "#31a354",
    "Cultures permanentes": "#8856a7",
    "Non attribué / Hors production (IAE, SNE)": "#7f7f7f",
}
couleurs_barres_grp = [palette_cat.get(c, "#7f7f7f") for c in stats_groupe["categorie_dominante"]]

bars_grp = ax1.barh(
    stats_groupe["libelle_groupe"],
    stats_groupe["surface_mha"],
    color=couleurs_barres_grp,
    edgecolor="none",
    height=0.75,
)
ax1.invert_yaxis()
ax1.set_xlabel("Surface déclarée (millions d'hectares)")
ax1.set_xlim(0, stats_groupe["surface_mha"].max() * 1.3)
ax1.set_title(f"Surface nationale des {len(stats_groupe)} groupes de cultures (`libelle_groupe`)", fontweight="bold", pad=12)

for bar, pct in zip(bars_grp, stats_groupe["part_sau_%"]):
    w = bar.get_width()
    ax1.text(
        w + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{w:.2f} M ha ({pct:.1f} %)",
        va="center",
        fontsize=8.5,
    )

handles_cat = [mpatches.Patch(color=col, label=cat) for cat, col in palette_cat.items()]
ax1.legend(handles=handles_cat, title="Catégorie dominante", loc="lower right", frameon=True, facecolor="white", fontsize=8.5)

# Sous-graphique 2 : Top 15 des cultures détaillées (libelle_culture)
y_cult = np.arange(len(top15_cult))
ax2.barh(y_cult, top15_cult.to_numpy(), color="#6baed6", height=0.72)

ax2.set_yticks(y_cult)
ax2.set_yticklabels(labels_courts, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel("Surface déclarée (millions d'hectares)")
ax2.set_xlim(0, top15_cult.max() * 1.3)
ax2.set_title(f"Top 15 des cultures détaillées (`libelle_culture` sur {df['libelle_culture'].nunique()})", fontweight="bold", pad=12)

for idx, tot in enumerate(top15_cult):
    ax2.text(tot + 0.08, idx, f"{tot:.2f} M ha ({100 * tot * 1e6 / df['surface_ha'].sum():.1f} %)", va="center", fontsize=8.5)

plt.tight_layout()
plt.show()

### Analyse de la spécialisation culturale

Les deux graphiques montrent une **forte concentration de la SAU** sur quelques productions :

1. **Hiérarchie des groupes de cultures (`libelle_groupe`)** :
   - Les **Prairies permanentes** arrivent en tête avec **7,56 millions d'ha** (**27,5 %** de la SAU), auxquelles s'ajoutent **2,10 millions d'ha** d'**Estives et landes** (**7,6 %**), **1,39 million d'ha** de **Prairies temporaires** (**5,1 %**) et **1,09 million d'ha** de **Fourrage** (**4,0 %**) : l'herbe et les fourrages couvrent plus de **44 % de la SAU**.
   - Parmi les grandes cultures, trois groupes dominent : le **Blé tendre** (**4,72 M ha**, **17,2 %**), le **Maïs grain et ensilage** (**2,57 M ha**, **9,4 %**) et l'**Orge** (**1,81 M ha**, **6,6 %**), suivis des oléagineux : **Colza** (**1,35 M ha**, **4,9 %**) et **Tournesol** (**0,81 M ha**, **3,0 %**).
   - Les **Vignes** occupent **0,60 M ha** (**2,2 %** de la SAU) et les **Autres cultures industrielles** (betterave notamment) **0,47 M ha** (**1,7 %**).

2. **Lecture pour l'exposition aux produits phytosanitaires (BNV-D)** :
   - Sur les **137 cultures**, les **15 premières couvrent 89,2 % de la SAU** (`24,5 M ha`).
   - L'usage des pesticides dépend du rapport local entre :
     - les cultures à **indice de fréquence de traitement (IFT) élevé** (*blé tendre d'hiver* `4,71 M ha`, *maïs* `2,55 M ha`, *orge d'hiver et de printemps* `1,82 M ha`, *colza d'hiver* `1,34 M ha`, *vigne* `0,60 M ha`, *betterave* `0,40 M ha`) ;
     - et les surfaces **peu ou pas traitées** (*prairies permanentes* `PPH` `7,56 M ha`, *surfaces pastorales* `SPH`/`SPL` `2,10 M ha`, *jachères* `JAC` `0,44 M ha`, *luzerne* `LUZ` `0,46 M ha`).

---

## 3. Disparités régionales : spécialisation des régions

Comment ces filières se répartissent-elles entre les **13 régions métropolitaines** ? Nous comparons la surface agricole de chaque région et son profil cultural.

In [ ]:
# Regroupement des 23 groupes de cultures en 6 grandes familles agronomiques lisibles pour le profil régional
FAMILLE_AGRONOMIQUE = {
    "Blé tendre": "Céréales à paille & Riz",
    "Orge": "Céréales à paille & Riz",
    "Autres céréales": "Céréales à paille & Riz",
    "Riz": "Céréales à paille & Riz",
    "Maïs grain et ensilage": "Maïs (grain & ensilage)",
    "Colza": "Oléagineux, Protéagineux & Indus.",
    "Tournesol": "Oléagineux, Protéagineux & Indus.",
    "Autres oléagineux": "Oléagineux, Protéagineux & Indus.",
    "Protéagineux": "Oléagineux, Protéagineux & Indus.",
    "Légumineuses à grains": "Oléagineux, Protéagineux & Indus.",
    "Plantes à fibres": "Oléagineux, Protéagineux & Indus.",
    "Autres cultures industrielles": "Oléagineux, Protéagineux & Indus.",
    "Canne à sucre": "Oléagineux, Protéagineux & Indus.",
    "Prairies permanentes": "Prairies permanentes & Estives",
    "Estives et landes": "Prairies permanentes & Estives",
    "Prairies temporaires": "Prairies temp. & Fourrages",
    "Fourrage": "Prairies temp. & Fourrages",
    "Vignes": "Vignes, Vergers & Cultures perm.",
    "Vergers": "Vignes, Vergers & Cultures perm.",
    "Fruits à coque": "Vignes, Vergers & Cultures perm.",
    "Oliviers": "Vignes, Vergers & Cultures perm.",
    "Gel (surfaces gelées sans production)": "Jachères & Divers",
    "Divers": "Jachères & Divers",
}
df["famille_agro"] = df["libelle_groupe"].map(FAMILLE_AGRONOMIQUE).fillna("Jachères & Divers")

# Statistiques régionales
stats_region = (
    df.groupby("region", as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_totale_ha=("surface_ha", "sum"),
        surface_moy_parcelle_ha=("surface_ha", "mean"),
    )
)
stats_region = stats_region.sort_values("surface_totale_ha", ascending=True)

# Profil cultural 100 % empilé par région (ordonné comme stats_region)
ordre_familles = [
    "Prairies permanentes & Estives",
    "Prairies temp. & Fourrages",
    "Vignes, Vergers & Cultures perm.",
    "Maïs (grain & ensilage)",
    "Céréales à paille & Riz",
    "Oléagineux, Protéagineux & Indus.",
    "Jachères & Divers",
]
couleurs_familles = {
    "Prairies permanentes & Estives": "#31a354",
    "Prairies temp. & Fourrages": "#a1d99b",
    "Vignes, Vergers & Cultures perm.": "#8856a7",
    "Maïs (grain & ensilage)": "#fd8d3c",
    "Céréales à paille & Riz": "#2b8cbe",
    "Oléagineux, Protéagineux & Indus.": "#feb24c",
    "Jachères & Divers": "#bdbdbd",
}

piv_reg_fam = (
    df.groupby(["region", "famille_agro"])["surface_ha"]
    .sum()
    .unstack(fill_value=0.0)
    .reindex(index=stats_region["region"], columns=ordre_familles, fill_value=0.0)
)
piv_reg_pct = piv_reg_fam.div(piv_reg_fam.sum(axis=1), axis=0) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7.2), sharey=True)

# Sous-graphique 1 : SAU de chaque région
bars_r = ax1.barh(stats_region["region"], stats_region["surface_totale_ha"] / 1e6, color="#6baed6", height=0.72)
ax1.set_xlabel("Surface agricole déclarée (millions d'ha)")
ax1.set_xlim(0, stats_region["surface_totale_ha"].max() / 1e6 * 1.3)
ax1.set_title("Surface agricole déclarée par région (RPG 2023)", fontweight="bold", pad=12)

for bar, moy in zip(bars_r, stats_region["surface_moy_parcelle_ha"]):
    w = bar.get_width()
    ax1.text(w + 0.05, bar.get_y() + bar.get_height() / 2, f"{w:.2f} M ha (parcelle moy. {moy:.1f} ha)", va="center", fontsize=8.5)

# Sous-graphique 2 : Profil cultural 100 % empilé de chaque région
cumul = np.zeros(len(piv_reg_pct))
for fam in ordre_familles:
    vals = piv_reg_pct[fam].to_numpy()
    ax2.barh(
        piv_reg_pct.index,
        vals,
        left=cumul,
        color=couleurs_familles[fam],
        edgecolor="white",
        linewidth=0.4,
        height=0.72,
        label=fam,
    )
    cumul += vals

ax2.set_xlim(0, 100)
ax2.set_xlabel("Part dans la SAU régionale (%)")
ax2.set_title("Spécialisation culturale des régions (% de la SAU régionale)", fontweight="bold", pad=12)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3, frameon=True, facecolor="white", fontsize=8.5)

plt.tight_layout()
plt.show()

### Analyse des disparités régionales : grandes plaines contre montagne et littoral

La surface agricole régionale (à gauche) et la composition culturale (à droite) font ressortir les grands équilibres territoriaux :

1. **Des régions de tailles agricoles très inégales** :
   - La **Nouvelle-Aquitaine** (**3,87 M ha**) et l'**Occitanie** (**3,65 M ha**) dominent, devant l'**Auvergne-Rhône-Alpes** (**3,05 M ha**) et le **Grand Est** (**2,98 M ha**). La **Provence-Alpes-Côte d'Azur** (**1,04 M ha**) et l'**Île-de-France** (**0,56 M ha**) ferment la marche.
   - La parcelle moyenne va de **1,9 ha** en Bretagne et **2,2 ha** en Occitanie et en Auvergne-Rhône-Alpes à **4,0 ha** en Centre-Val de Loire et **4,5 ha** en Île-de-France.

2. **La spécialisation régionale dessine la géographie des pressions agricoles** :
   - En **Île-de-France**, **Centre-Val de Loire**, **Hauts-de-France** et **Grand Est**, les **céréales à paille**, les **oléagineux, protéagineux et cultures industrielles** et le **maïs** occupent l'essentiel de la SAU : ce sont des systèmes d'openfield très consommateurs d'herbicides et de fongicides.
   - En **Bretagne** et **Pays de la Loire**, on retrouve la polyculture-élevage de l'Ouest : forte part de **maïs (grain et ensilage)** et de **prairies temporaires et fourrages** en rotation avec les céréales.
   - En **Provence-Alpes-Côte d'Azur**, **Auvergne-Rhône-Alpes** et **Occitanie**, les **prairies permanentes et estives** dominent, avec en Provence et en Occitanie une part notable de **vignes, vergers et cultures permanentes**.

---

## 4. Cartographie départementale (choroplèthes)

Pour préparer la jointure à la **maille départementale (`code_dep`)** prévue par le [contrat de données](../../docs/contrat_donnees.md), nous représentons trois cartes choroplèthes sur les 94 départements du périmètre (sans la Corse).

In [ ]:
# Agrégation par département (maille de jointure du contrat de données)
GROUPES_FORTE_PRESSION = {
    "Blé tendre", "Maïs grain et ensilage", "Orge", "Autres céréales",
    "Colza", "Tournesol", "Autres oléagineux", "Protéagineux",
    "Plantes à fibres", "Autres cultures industrielles",
    "Vignes", "Vergers", "Fruits à coque", "Oliviers",
}
df["culture_pression_phyto"] = df["libelle_groupe"].isin(GROUPES_FORTE_PRESSION)
df["surface_pression_ha"] = np.where(df["culture_pression_phyto"], df["surface_ha"], 0.0)
df["surface_herbe_ha"] = np.where(df["libelle_groupe"].isin({"Prairies permanentes", "Estives et landes"}), df["surface_ha"], 0.0)

stats_dep = (
    df.groupby(["code_dep", "nom_dep"], as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_totale_ha=("surface_ha", "sum"),
        surface_moy_ha=("surface_ha", "mean"),
        surface_pression_ha=("surface_pression_ha", "sum"),
        surface_herbe_ha=("surface_herbe_ha", "sum"),
    )
)
stats_dep["part_herbe_%"] = 100 * stats_dep["surface_herbe_ha"] / stats_dep["surface_totale_ha"]
stats_dep["part_cultures_intensives_%"] = 100 * stats_dep["surface_pression_ha"] / stats_dep["surface_totale_ha"]

# Contours des 94 départements du périmètre (sans la Corse), en Lambert 93
deps_metro = recuperer_departements().to_crs(2154)
carte_dep = deps_metro.merge(stats_dep.drop(columns="nom_dep"), on="code_dep", how="left")

# Tracé des 3 cartes choroplèthes départementales côte à côte (bornes ajustées hors micro-département 75 = 3 parcelles)
fig, axes = plt.subplots(1, 3, figsize=(19, 6.8))

configurations_cartes = [
    (
        "part_cultures_intensives_%",
        "YlOrRd",
        0.0,
        95.0,
        "1. Part des grandes cultures, vignes & vergers\ndans la SAU départementale (%)",
        "% de la SAU",
    ),
    (
        "part_herbe_%",
        "YlGn",
        0.0,
        100.0,
        "2. Part des prairies permanentes & estives\ndans la SAU départementale (%)",
        "% de la SAU en herbe",
    ),
    (
        "surface_moy_ha",
        "PuBu",
        0.8,
        7.0,
        "3. Surface moyenne d'une parcelle (`surface_ha`)\npar département (en ha)",
        "Surface moyenne en ha (plafonnée à 7 ha)",
    ),
]

for ax, (col, cmap, vmin, vmax, titre, label_cb) in zip(axes, configurations_cartes):
    carte_dep.plot(
        column=col,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidth=0.45,
        edgecolor="#333333",
        legend=True,
        legend_kwds={"label": label_cb, "orientation": "horizontal", "shrink": 0.78, "pad": 0.03},
        ax=ax,
    )
    ax.set_title(titre, fontweight="bold", fontsize=11, pad=10)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

### Analyse spatiale des cartes départementales

- **Carte 1 (part des grandes cultures, vignes et vergers)** : le grand arc céréalier (**Beauce, Brie, Picardie, Champagne crayeuse, Poitou-Charentes, plaine d'Alsace, vallée de la Garonne**) consacre l'essentiel de sa SAU aux cultures à forte pression phytosanitaire (près de **89 %** en Eure-et-Loir et en Seine-et-Marne, plus de **80 %** dans la Marne, l'Oise, le Val-d'Oise et l'Essonne), à l'opposé de la diagonale herbagère et montagnarde (**Massif central, Jura, Alpes, Pyrénées**).
- **Carte 2 (part des prairies permanentes et estives)** : c'est presque l'image inversée de la carte 1, avec plus de **80 %** de la SAU en herbe en Savoie, dans les Hautes-Alpes, le Cantal, l'Ariège ou la Lozère, contre moins de **5 %** en Essonne, Seine-et-Marne, Eure-et-Loir ou Marne.
- **Carte 3 (surface moyenne des parcelles)** : on retrouve l'openfield du Bassin parisien, de la Lorraine et du Berry (parcelles moyennes de **4 à 5,5 ha**) et les grands parcours pastoraux des Alpes-Maritimes, face au parcellaire fin (moins de **2 ha** en moyenne) de la Bretagne, de la façade atlantique et des bassins viticoles.

---

### Q&R
- **Comment se structurent les surfaces agricoles du RPG 2023 selon les catégories, groupes et cultures ?**
  Le RPG 2023 compte **9 742 907 parcelles** pour **27 457 975 ha** sur 94 départements. L'espace agricole est dominé par les **Terres arables** (**16,69 M ha**, **60,8 %** de la SAU) et les **Prairies permanentes** (**9,68 M ha**, **35,3 %**), tandis que les **Cultures permanentes** (vignes, vergers, oliviers) couvrent **0,86 M ha** (**3,1 %** de la SAU mais **7,6 %** des parcelles).

### Principaux résultats
- **Asymétrie de la taille des parcelles (`surface_ha`)** : la parcelle médiane fait **1,24 ha** (Q25 = **0,39 ha**, Q75 = **3,21 ha**) pour une moyenne de **2,83 ha** et un 95e centile à **10,38 ha** (médiane de **2,00 ha** en Terres arables, **1,21 ha** en Prairies permanentes et **0,63 ha** en Cultures permanentes).
- **Forte concentration culturale** : sur 137 cultures, les **15 premières couvrent 89,2 % de la SAU**. Les **Prairies permanentes** (**7,56 M ha**, 27,5 %), le **Blé tendre** (**4,72 M ha**, 17,2 %) et le **Maïs grain et ensilage** (**2,57 M ha**, 9,4 %) en rassemblent plus de la moitié, devant les **Estives et landes** (**2,10 M ha**), l'**Orge** (**1,81 M ha**), les **Prairies temporaires** (**1,39 M ha**) et le **Colza** (**1,35 M ha**).
- **Clivage territorial net** entre les plaines de grandes cultures du Nord et du Centre (Île-de-France, Centre-Val de Loire, Hauts-de-France, Grand Est) et les régions d'herbe et de montagne (Auvergne-Rhône-Alpes, Provence-Alpes-Côte d'Azur), l'Ouest étant dominé par la polyculture-élevage.

### Pistes pour la suite
- **Indicateurs pour la jointure** : lors du croisement avec les achats de pesticides (**BNV-D**) et les indices biologiques (**Hub'Eau**), ne pas se limiter à la SAU totale du département : inclure la **part de cultures à IFT élevé** (céréales, colza, maïs, betterave, pomme de terre, vigne) et la **part d'herbe** comme variable modératrice de la pression phytosanitaire.
- **Suite** : relier `data/processed/parcelles_rpg_2023.parquet` aux achats de pesticides par département (`code_dep`), conformément au contrat de données.